# Notebook 01 – Datenbereinigung und Exploration

Dieses Notebook bildet die Grundlage für alle weiteren Analysen im Projekt.
Es lädt den Rohdatensatz, bereinigt ihn und exportiert eine saubere CSV-Datei,
auf die alle folgenden Notebooks aufbauen.

**Projektziel:** Online-Retail-Transaktionen auswerten und typische Analysefragen beantworten:
- Wie viele gültige Verkäufe enthält der Datensatz?
- Welche Länder und Produkte sind besonders umsatzstark?
- Wie entwickelt sich der Umsatz über die Monate?
- Wie hoch ist der Anteil an Stornierungen?

## 1. Bibliotheken importieren

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Daten einlesen

Die Rohdatei hat einige Besonderheiten, die beim Einlesen berücksichtigt werden müssen:
- Spalten sind durch Semikolon getrennt (`sep=";"`)
- Dezimalzahlen verwenden ein Komma statt Punkt (`decimal=","`)
- Die erste Zeile ist eine Beschreibung, keine Kopfzeile (`skiprows=1`)
- Das Encoding enthält ein BOM-Zeichen (`encoding="utf-8-sig"`)
- Mindestens eine korrupte Zeile im Rohdatensatz wird übersprungen (`on_bad_lines="skip"`)

In [ ]:
df = pd.read_csv(
    "../data/online_retail_data.csv",
    sep=";",
    skiprows=1,
    decimal=",",
    parse_dates=["InvoiceDate"],
    dayfirst=True,
    encoding="utf-8-sig",
    on_bad_lines="skip"
)

df.head()

## 3. Erste Datenexploration

Wir verschaffen uns einen ersten Überblick über Größe, Datentypen und fehlende Werte.

In [ ]:
# Anzahl Zeilen und Spalten
df.shape

In [ ]:
# Datentypen und nicht-null Einträge pro Spalte
df.info()

In [ ]:
# Fehlende Werte pro Spalte
df.isna().sum()

In [ ]:
# Statistische Kennzahlen aller Spalten
df.describe(include="all")

## 4. Neue Spalten berechnen

Für die Analyse werden drei neue Spalten erstellt:
- `TotalPrice`: Umsatz pro Zeile (Menge × Stückpreis)
- `InvoiceMonth`: Monat der Rechnung (für Zeitreihenanalysen)
- `IsCancellation`: Kennzeichnung von Stornierungen anhand des Präfix `C` in der Rechnungsnummer

In [ ]:
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]
df["InvoiceMonth"] = df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
df["IsCancellation"] = df["InvoiceNo"].astype(str).str.startswith("C")

df[["InvoiceNo", "Quantity", "UnitPrice", "TotalPrice", "InvoiceDate", "InvoiceMonth", "IsCancellation"]].head()

## 5. Spaltenauswahl (Projektion)

Wir wählen für die weitere Analyse nur die relevanten Spalten aus.

In [ ]:
# Einzelne Spalte anzeigen
df["Country"]

In [ ]:
# Mehrere Spalten gezielt auswählen
df[["InvoiceNo", "Description", "Quantity", "UnitPrice", "TotalPrice", "Country"]].head(10)

## 6. Daten filtern (Restriction)

Für alle weiteren Analysen werden nur gültige Verkaufszeilen verwendet.
Ausgeschlossen werden:
- Stornierungen (IsCancellation == True)
- Zeilen mit negativer Menge oder negativem Preis
- Zeilen ohne Kundennummer (für kundenbezogene Analysen erforderlich)

In [ ]:
sales = df.loc[
    (df["IsCancellation"] == False) &
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0) &
    (df["CustomerID"].notna())
].copy()

print(f"Gültige Verkaufszeilen: {sales.shape[0]:,} von {df.shape[0]:,} gesamt")
sales.head()

## 7. Wichtige Kennzahlen

Diese Kennzahlen geben einen kompakten Überblick über den bereinigten Datensatz.

In [ ]:
kennzahlen = {
    "Anzahl Zeilen gesamt": len(df),
    "Anzahl gültige Verkaufszeilen": len(sales),
    "Anzahl eindeutige Rechnungen": sales["InvoiceNo"].nunique(),
    "Anzahl eindeutige Kund:innen": sales["CustomerID"].nunique(),
    "Anzahl Länder": sales["Country"].nunique(),
    "Gesamtumsatz (€)": sales["TotalPrice"].sum(),
    "Durchschnittlicher Warenkorb pro Rechnung (€)": sales.groupby("InvoiceNo")["TotalPrice"].sum().mean()
}

pd.Series(kennzahlen)

## 8. Analyse nach Ländern

Welche Länder erzeugen den meisten Umsatz?

In [ ]:
umsatz_laender = (
    sales.groupby("Country")["TotalPrice"]
    .sum()
    .sort_values(ascending=False)
)

umsatz_laender.head(10)

In [ ]:
umsatz_laender.head(10).plot(kind="bar", figsize=(12, 6))
plt.title("Top 10 Länder nach Umsatz")
plt.xlabel("Land")
plt.ylabel("Umsatz (€)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../output/01_top10_laender_umsatz.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Analyse nach Produkten

Welche Produkte wurden am häufigsten verkauft?

In [ ]:
top_produkte = (
    sales.groupby("Description")["Quantity"]
    .sum()
    .sort_values(ascending=False)
)

top_produkte.head(10)

In [ ]:
top_produkte.head(10).sort_values().plot(kind="barh", figsize=(10, 6))
plt.title("Top 10 Produkte nach verkaufter Menge")
plt.xlabel("Verkaufte Menge")
plt.ylabel("Produkt")
plt.tight_layout()
plt.savefig("../output/01_top10_produkte_menge.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Umsatzentwicklung über Zeit

Der monatliche Umsatzverlauf zeigt saisonale Trends und Ausreißer.

In [ ]:
monatsumsatz = (
    sales.groupby("InvoiceMonth")["TotalPrice"]
    .sum()
    .sort_index()
)

monatsumsatz

In [ ]:
monatsumsatz.plot(kind="line", marker="o", figsize=(12, 6))
plt.title("Monatlicher Umsatz")
plt.xlabel("Monat")
plt.ylabel("Umsatz (€)")
plt.grid(True)
plt.tight_layout()
plt.savefig("../output/01_monatlicher_umsatz.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Stornierungen analysieren

Stornierungen zeigen an, wie viele Transaktionen nachträglich rückgängig gemacht wurden.

In [ ]:
stornos = df.loc[df["IsCancellation"] == True].copy()

storno_anzahl = stornos.shape[0]
storno_anteil = storno_anzahl / df.shape[0] * 100

print(f"Stornierungen: {storno_anzahl:,} ({storno_anteil:.2f}% aller Zeilen)")

In [ ]:
df["IsCancellation"].value_counts().plot(kind="bar", figsize=(6, 4))
plt.title("Normale Rechnungen vs. Stornierungen")
plt.xlabel("Ist Stornierung?")
plt.ylabel("Anzahl Zeilen")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../output/01_stornierungen.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Bereinigte Daten exportieren

Der gefilterte Verkaufsdatensatz wird als CSV gespeichert.
Diese Datei wird in den folgenden Notebooks (02, 03, 04) als Ausgangspunkt verwendet.

In [ ]:
sales.to_csv("../data/online_retail_data_cleaned.csv", index=False)
print("Datei gespeichert: ../data/online_retail_data_cleaned.csv")

## 13. Zusammenfassung

Aus der Exploration lassen sich folgende Erkenntnisse ableiten:

- **United Kingdom** macht den größten Teil des Umsatzes aus
- Ca. **25 % der Zeilen** haben keine Kundennummer und werden für kundenbezogene Analysen ausgeschlossen
- Die **Stornierungsrate** beträgt ca. 1,7 % der Gesamttransaktionen
- Der Umsatz zeigt **saisonale Schwankungen** mit einem Peak gegen Jahresende

Diese Erkenntnisse fließen direkt in die Preprocessing-Entscheidungen der ML-Notebooks ein.